# Final Project - ST - 554 Big data analysis

## Author: Yefrid Cordoba

### Fitting model

In this section we are going to use the whole data from the file `'power_ml_data.csv'` for the fitting of the models, using first a principal component analysis to pick the most important features, which are going to fe fed to a elastic net model crossvalidation.

We import the modules that are going to be used

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, VectorAssembler, StandardScaler, OneHotEncoder, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

And create the spark session

In [2]:
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/26 23:14:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/26 23:14:39 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Now we import the data as a pandas dataframe, and then we are going to convert it to a spark dataframe.

In [3]:
power_fit = pd.read_csv('power_ml_data.csv')

In [4]:
#We convert to spark dataframe
power_fit = spark.createDataFrame(power_fit)

In [5]:
power_fit.schema

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True)])

In [6]:
power_fit.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

We are going to transformthe hour column from a `LongType()` to a `DoubleType()`.

In [7]:
type_double = SQLTransformer(
    statement="""
        SELECT * EXCEPT (Hour), 
        CAST(Hour AS DOUBLE) AS Hour 
        FROM __THIS__
    """
)

Also, we want to binarize the Hour column based on it being less than 6.5 or not.

In [8]:
 bin_hour = SQLTransformer( 
    statement = """
            SELECT *,
            CASE WHEN Hour < 6.5 THEN 1 ELSE 0 END AS bin_Hour
            FROM __THIS__
            """
 )

In [9]:
bin_hour.transform(type_double.transform(power_fit)).show(20)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+--------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|bin_Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+--------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1| 0.0|       1|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1| 0.0|       1|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1| 0.0|       1|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1| 0.0|       1|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 1

Now we ar going to create a column with the `features`, and rename the response column as `label`.

In [10]:
feature_cols = ['Temperature', 'Humidity', 'Wind_Speed', 'General_Diffuse_Flows', 'Diffuse_Flows']
pca_assem = VectorAssembler(inputCols = feature_cols,\
                         outputCol = 'feature_PC')

In [11]:
rename_col = SQLTransformer( 
    statement = """
            SELECT *, Power_Zone_3 as label
            FROM __THIS__
            """
    )

Also as PCA is sensitive to magnitude of the feautres, we are going to standarize the feature column, process that is going to be repeated when setting up the feautres for the elastic net regression.

In [12]:
scaler = StandardScaler(
    inputCol='feature_PC', 
    outputCol='scaled_features_PC',
    withMean=True,
    withStd=True
)

We use the OneHotEcoder for the month variable, setting droplast = True, so it recognizes that when all the codes are 0, then it is the last month.

In [13]:
month_encoder = OneHotEncoder(
    inputCol='Month',
    outputCol="Month_Encoded",
    dropLast=True
)

In [14]:
month_encoder.fit(power_fit).transform(power_fit).show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+--------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour| Month_Encoded|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+--------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|(12,[1],[1.0])|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|(12,[1],[1.0])|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|(12,[1],[1.0])|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|(12,[1],[1.0])|
|      5.921|    75.7|     0.081|                0.048|

With the standarized features, we set up the stimator for the PCA, using 2 principal components.

In [15]:
pca = PCA(
    k=2,
    inputCol='scaled_features_PC', 
    outputCol='pca_features'
)

Now we create the transformer for the standard scaling of the missing continuous variables that were not scaled for the PCA.

In [16]:
feature_p = ['Power_Zone_1', 'Power_Zone_2']
pz_assem = VectorAssembler(inputCols = feature_p,\
                         outputCol = 'feature_pz')

In [17]:
scaler_pz = StandardScaler(
    inputCol='feature_pz', 
    outputCol='scaled_features_pz',
    withMean=True,
    withStd=True
)

And finally we can put all the features(from PCA, scaled and binary) in one column that is going to be used when fitting the Elastic net.

In [18]:
feature_assem = VectorAssembler(inputCols = ['pca_features', 'scaled_features_pz', 'bin_Hour', 'Month_Encoded'],\
                         outputCol = 'features')

We can add linear regression as one of the transformations for our pipeline, which later is going to be used for the cross validator.

In [19]:
lr = LinearRegression()

Now that we have all the transformations we can fit our pipeline.

In [20]:
pipeline = Pipeline(stages = [type_double, bin_hour, pca_assem, rename_col, scaler, month_encoder, pca, pz_assem, scaler_pz, feature_assem, lr])

In [21]:

paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()
crossval = CrossValidator(estimator = pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(metricName='rmse'),
                          numFolds=5) #now use .fit() method on crossval!

In [24]:
cvModel = crossval.fit(power_fit)

26/04/26 23:30:44 WARN Instrumentation: [5b694773] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 23:30:44 WARN Instrumentation: [5b694773] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/26 23:30:45 WARN Instrumentation: [af6d2819] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 23:30:45 WARN Instrumentation: [af6d2819] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/26 23:30:46 WARN Instrumentation: [0af76f59] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 23:30:46 WARN Instrumentation: [0af76f59] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/26 23:30:48 WARN Instrumentation: [8fed5c34] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 23:30:48 WARN Instrumentation: [8fed5c34] Cholesky solv

Now we can get the best regression parameter.

In [30]:
cvModel.bestModel.stages[-1].getRegParam()

0.9

And the best Elastic net parameter from the cross validation process.

In [31]:
cvModel.bestModel.stages[-1].getElasticNetParam()

0.05

Also, we report the cross-validation error, which is the lowest RMSE value for the combination (optimal parameters).

In [33]:
min(cvModel.avgMetrics)

2175.243124943382